In [ ]:
import scanpy as sc #основная библиотека по работе с scRNA-seq на python. Аналог Seurat R
import pandas as pd #манипуляции с таблицами 
import seaborn as sns #библиотека для визуализации
import matplotlib.pyplot as plt #библиотека для визуализации
import numpy as np #библиотека для работы с числовыми матрицами

In [ ]:
import warnings
warnings.filterwarnings('ignore') 

In [ ]:
sc.settings.figdir = "./figures"

In [ ]:
from scipy.io import mmread
import scipy.sparse as sp
import tarfile

with tarfile.open("Data_Lee2020_Colorectal.tar.gz", "r:gz") as tar:
    f = tar.extractfile("Data_Lee2020_Colorectal/Exp_data_UMIcounts.mtx")
    matrix = mmread(f).tocsc()  #читаем матрицу из архива

In [ ]:
genes = pd.read_csv('Genes.txt', header = None, names=["gene_id"]) 
genes.head()

In [ ]:
cells = pd.read_csv('Cells.csv')
cells.head()

In [ ]:
meta = pd.read_csv('Meta-data.csv')
meta.head()

In [ ]:
adata = sc.AnnData(X=matrix.transpose().toarray(), var=genes, obs=cells) #создаем AnnData, транспонируя матрицу, добавляя данные о генах и клетках
adata

In [ ]:
columns_of_interes = ['sex', 'age','disease_extent', 'AJCC_stage', 'site', 'genetic_hormonal_features']
for i in columns_of_interes:
    map_dict = dict(zip(meta['sample'], meta[i]))
    adata.obs[i] = adata.obs['sample'].map(map_dict)

adata.obs.genetic_hormonal_features = adata.obs.genetic_hormonal_features.map({'wtKRAS, mss':'wtKRAS','wtKRAS , mss':'wtKRAS',
                                                                              'mKRAS, mss':'mKRAS'})

In [ ]:
adata.obs = adata.obs.rename(columns={'genetic_hormonal_features': 'KRAS_status'})

In [ ]:
adata.obs.KRAS_status.value_counts()

In [ ]:
adata.var_names = adata.var.gene_id
adata.var_names_make_unique()

In [ ]:
adata.obs

In [ ]:
adata.var

In [ ]:
adata.obs.cell_type.value_counts()

In [ ]:
adata.obs['sample'].value_counts().plot.barh(figsize = (4,3))

<h1>QC</h1>

In [ ]:
"""%%time
sc.external.pp.scrublet(adata, threshold=0.25)
sc.external.pl.scrublet_score_distribution(adata)"""

In [ ]:
"""adata.obs.predicted_doublet.value_counts()
adata = adata[adata.obs.predicted_doublet == False]"""

In [ ]:
# mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
# ribosomal genes
adata.var['ribo'] = adata.var_names.str.startswith(("RPS","RPL"))
# hemoglobin genes.
adata.var['hb'] = adata.var_names.str.contains(("^HB[^(P)]"))
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], percent_top=None, log1p=False, inplace=True) #считаем метрики qc

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, frameon=True, 
                     vector_friendly=False, fontsize=12, figsize=None, 
                     color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='png2x')
plt.rcParams['figure.figsize'] = [6,2]
plt.rcParams["savefig.dpi"] = 300
sc.pl.violin(adata, ['pct_counts_mt'],
             jitter=0.2, groupby = 'sample', rotation= 90)
plt.rcParams['figure.figsize'] = [14,4]
sc.pl.violin(adata, ['pct_counts_ribo'],
             jitter=0.2, groupby = 'sample', rotation= 90)

In [ ]:
plt.rcParams['figure.figsize'] = [14,4]
plt.rcParams["savefig.dpi"] = 300
sc.pl.violin(adata, ['total_counts'], size = 1.3,
             jitter=0.2, groupby = 'sample', rotation= 90)

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)  # Удаление клеток с менее чем 200 генами
sc.pp.filter_genes(adata, min_cells=3)  # Удаление генов, экспрессируемых менее чем в 3 клетках

In [ ]:
# Apply additional filtering
adata = adata[adata.obs['total_counts'] > 500, :]#Убираем клетки, у которых мало РНК — это могут быть мертвые клетки или клетки с плохим качеством захвата.
adata = adata[adata.obs['total_counts'] < 25000, :]#Условие < 25000 удаляет клетки с чрезмерно высокой экспрессией, которые могут быть двойными клетками (doublets) или техническим шумом.
adata = adata[adata.obs['n_genes_by_counts'] < 5000, :]
adata = adata[adata.obs['pct_counts_mt'] < 20, :]

In [ ]:
ribo_genes  = adata.var_names.str.startswith(("RPL","RPS")) #Поиск генов рибосом и MALAT1 и создание булевого массива для них
malat_gene = adata.var_names.str.startswith("MALAT1")

remove = np.add(malat_gene, ribo_genes) #np.add для булевых массивов работает как логическое ИЛИ, т.е. remove[i] = True, если ген i — рибосомный или MALAT1.
keep = np.invert(remove) #np.invert меняет True ↔ False, чтобы получить маску для сохранения генов, не попадающих в remove.

adata = adata[:,keep] #Оставляем только те гены, которые не являются рибосомными и не MALAT1.

print(adata.n_obs, adata.n_vars)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20,palette="Blues", width=.3)#отбираем самые высокоэкспресированные гены

In [ ]:
plt.rcParams['figure.figsize'] = [4,4]
sc.pl.scatter(adata, x='total_counts', size = 0.5, y='n_genes_by_counts')

In [ ]:
sns.histplot(adata.to_df().sum(1))

<H1>Нормализация<H1>      

In [ ]:
adata.raw = adata.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes = 3000)#отбираем самые высоковариабельные гены

In [ ]:
plt.rcParams['figure.figsize'] = [4,4]
sc.pl.highly_variable_genes(adata)

In [ ]:
adata = adata[:, adata.var.highly_variable].copy()
adata.write('adata_after_normal.h5ad')

In [ ]:
adata = sc.read('adata_after_normal.h5ad')

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.set_figure_params(dpi=100)
sc.pl.pca_variance_ratio(adata, n_pcs=40, log=True)

In [ ]:
plt.rcParams['figure.figsize'] = [4,4]
sc.pl.pca(adata, color = ['sample', 'cell_type'], wspace = 0.6)

In [ ]:
plt.rcParams['figure.figsize'] = [4,4]
sc.pl.umap(adata, color = ['sample', 'cell_type'], wspace = 0.6)

<h1>Интеграция</h1>

### Harmony

In [ ]:
adata_harmony = adata.copy()

In [ ]:
sc.external.pp.harmony_integrate(adata_harmony, key = 'sample')

In [ ]:
adata_harmony.obsm['X_pca'] = adata_harmony.obsm['X_pca_harmony']
sc.pp.neighbors(adata_harmony, n_pcs=30)
sc.tl.umap(adata_harmony)

In [ ]:
sc.tl.leiden(adata_harmony, resolution=0.5, key_added='leiden')

In [ ]:
plt.rcParams['figure.figsize'] = [5,5]
sc.pl.umap(adata_harmony, color = ['sample', 'cell_type'], ncols=1, wspace = 1)

In [ ]:
adata = adata_harmony

In [ ]:
adata.write('adata_after_integration.h5ad')

<H1>Поиск маркерных генов и аннотация</H1>

In [ ]:
adata = sc.read('adata_after_integration.h5ad')

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type'], ncols=1, wspace = 1)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale='var')

In [ ]:
plt.rcParams['figure.figsize'] = [4,4]
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
sc.pl.umap(adata, color=["LGR5", "ASCL2", "EPHB2", "OLFM4", "EPCAM", "KRT19", "CDX2", "CEACAM5", "FABP1", "MUC5AC", "TFF3", "MARCKSL1", "TGFBI", "COX5B", "LAMC2", "MKI67", "DSTN", "YWHAZ", "OCIAD2", "SPINT2"])

In [ ]:
import celltypist
from celltypist import models

In [ ]:
model = models.Model.load('Human_Colorectal_Cancer.pkl')
predictions = celltypist.annotate(adata, model=model, majority_voting=True)

In [ ]:
adata = predictions.to_adata()

In [ ]:
plt.rcParams['figure.figsize'] = [6,6]
sc.pl.embedding(adata, 
                basis = 'X_umap', 
                color = 'majority_voting',
               legend_fontsize = 10,
               legend_fontweight = 10)


In [ ]:
plt.rcParams['figure.figsize'] = [8,8]
sc.pl.umap(adata, color = ['majority_voting'], legend_loc='on data', ncols=1, wspace = 0.6, legend_fontsize=6, save='_majority_voting.png')


In [ ]:
sc.tl.rank_genes_groups(adata, 'majority_voting')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=3, standard_scale='var', save='_dotplot_annotation.png')

In [ ]:
adata = adata.copy()

rename_map = {
    "CD19+CD20+ B": "B cells",
    "Myofibroblasts": "myCAF_iCAF_apCAF",
    "SPP1+": "TAM_M2",
    "Pro-inflammatory": "TAM_M1",
    "CMS1": "Tumor cells",
    "CMS2": "Tumor cells", 
    "CMS3": "Tumor cells",
    "CMS4": "Tumor cells",
}

new_categories = ['myCAF', 'iCAF', 'apCAF']
adata.obs["cell_type_refined"] = adata.obs["majority_voting"].replace(rename_map)

if hasattr(adata.obs['cell_type_refined'], 'cat'):
    adata.obs['cell_type_refined'] = adata.obs['cell_type_refined'].cat.add_categories(new_categories)

def present(genes):
    return [g for g in genes if g in adata.var_names]

gene_sets = {
    "B cells": ["CD19", "MS4A1", "CD79A", "CD79B", "CD74", "HLA-DRA", "IGHM", "IGHD"],
    "myCAF": ["ACTA2", "TAGLN", "MYL9", "COL1A1", "COL1A2", "TPM2", "POSTN"],
    "iCAF": ["IL6", "CXCL12", "CCL11", "FAP", "LIF", "HGF", "CXCL14", "PDGFRA", "IL1R1"],
    "apCAF": ["HLA-DRA", "HLA-DRB1", "CD74", "CIITA"],
    "TAM_M1": ["IL1B", "TNF", "CXCL9", "CXCL10", "NFKBIA", "CCL5"],
    "TAM_M2": ["CD163", "MRC1", "C1QA", "C1QB", "C1QC", "APOE", "CCL18"],
}

gene_sets_present = {k: present(v) for k, v in gene_sets.items()}
gene_sets_present = {k: v for k, v in gene_sets_present.items() if len(v) > 0}

for name, genes in gene_sets_present.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{name}_score", use_raw=False)

b_mask = adata.obs["majority_voting"].eq("CD19+CD20+ B")
caf_mask = adata.obs["majority_voting"].str.contains("Stromal|Myofibroblast", na=False)
mac_mask = adata.obs["majority_voting"].eq("SPP1+")| adata.obs["majority_voting"].eq("Pro-inflammatory")

adata.obs.loc[b_mask, "cell_type_refined"] = "B cells"

caf_scores = adata.obs.loc[caf_mask, [f"{x}_score" for x in ["myCAF", "iCAF", "apCAF"] if f"{x}_score" in adata.obs.columns]]
if caf_scores.shape[1] > 0:
    caf_best = caf_scores.idxmax(axis=1).str.replace("_score", "", regex=False)
    adata.obs.loc[caf_mask, "cell_type_refined"] = caf_best.values

mac_scores = adata.obs.loc[mac_mask, [f"{x}_score" for x in ["TAM_M1", "TAM_M2"] if f"{x}_score" in adata.obs.columns]]
if mac_scores.shape[1] > 0:
    mac_best = mac_scores.idxmax(axis=1).str.replace("_score", "", regex=False)
    adata.obs.loc[mac_mask, "cell_type_refined"] = mac_best.values

In [ ]:
plt.rcParams['figure.figsize'] = [8,8]
sc.pl.umap(adata, color=["cell_type_refined"], wspace=0.8, legend_fontsize=8, save= '_umap_annotation.png')

In [ ]:
groups_to_test = [g for g in adata.obs['cell_type_refined'].unique() 
                  if (adata.obs['cell_type_refined'] == g).sum() >= 2]

# Запускаем только для них
sc.tl.rank_genes_groups(adata, 
                        groupby='cell_type_refined', 
                        groups=groups_to_test)

In [ ]:
adata.obs['cell_type_refined'] = adata.obs['cell_type_refined'].cat.remove_unused_categories()

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=3, standard_scale='var')

In [ ]:
sc.pl.umap(adata, color=[
    'CD3D', 'CD8A', 'FOXP3',   # T cells
    'MS4A1', 'CD19',                 # B
    'CD68', 'CD163',          # macrophages
    'COL1A1', 'ACTA2',        # CAF
    'EPCAM',                  # epithelial
    'PECAM1',                 # endothelial
    'NACA', 'COX6A1', 'TACSTD2'         #tumor
], save='_umap_genes.png')

In [ ]:
adata.write('adata_after_annotation.h5ad')

<H1>Клеточный состав</H1>

In [ ]:
adata = sc.read("adata_after_annotation.h5ad")

In [ ]:
adata.obs

In [ ]:
def item_series(item, indexed=None):
    if indexed is not None:
        if hasattr(indexed, 'index'):
            return pd.Series([item]*len(indexed), index=indexed.index)
        elif type(indexed) is int and indexed > 0:
            return pd.Series([item]*indexed, index=np.arange(indexed))
    return pd.Series()

def pivot_vectors(vec1, vec2):
    name1 = str(vec1.name)
    name2 = str(vec2.name)

    if name1 == name2:
        name1 += '_1'
        name2 += '_2'

    sub_df = pd.DataFrame({name1: vec1,
                           name2: vec2})
    fill_dict = {}
    sub_df.fillna(value=fill_dict, inplace=True)
    sub_df = sub_df.assign(N=item_series(1, sub_df))

    return pd.pivot_table(data=sub_df, columns=name1,
                          index=name2, values='N', aggfunc=sum).fillna(0).astype(int)

In [ ]:
cell_content = pivot_vectors(adata.obs['sample'], adata.obs.cell_type)

In [ ]:
cell_content_many_cells =  pivot_vectors(adata.obs['sample'], adata.obs.cell_type_refined)

In [ ]:
def convert_to_percentage(df):
    return df.div(df.sum()) * 100

In [ ]:
cell_percent = convert_to_percentage(cell_content)
cell_content_many_cells_percent = convert_to_percentage(cell_content_many_cells)

In [ ]:
plt.rcParams['figure.figsize'] = [8,3]
cell_content.T.plot(kind='bar', stacked=True, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2'])
plt.legend(loc = 1, bbox_to_anchor = (1.4,1), frameon = False, prop = {'weight':'bold'})
plt.title('total cells by sample')

plt.rcParams['figure.figsize'] = [8,3]
cell_percent.T.plot(kind='bar', stacked=True, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']) 
plt.legend(loc = 1, bbox_to_anchor = (1.4,1), frameon = False, prop = {'weight':'bold'})
plt.title('cells by sample, %')

In [ ]:
plt.rcParams['figure.figsize'] = [8,10]
cell_content_many_cells_percent.T.plot(kind='bar', stacked=True, color=['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#8c564b',
 '#e377c2',
 '#b5bd61',
 '#17becf',
 '#aec7e8',
 '#ffbb78',
 '#98df8a',
 '#ff9896',
 '#c5b0d5',
 '#c49c94',
 '#f7b6d2',
 '#dbdb8d',
 '#9edae5',
 '#ad494a',
 '#8c6d31'])
plt.legend(loc = 1, bbox_to_anchor = (1.6,1), frameon = False, prop = {'weight':'bold'})
plt.title('% cells by sample')

In [ ]:
adata.obs.KRAS_status = adata.obs.KRAS_status.astype('category')

<H1>scCODA</H1>

In [ ]:
import pertpy as pt

In [ ]:
sccoda_model = pt.tl.Sccoda()
sccoda_data = sccoda_model.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell_type_refined", 
    sample_identifier="sample",            
    covariate_obs=["KRAS_status"],           
)

In [ ]:
pt.pl.coda.boxplots(
    sccoda_data,
    modality_key="coda",
    feature_name="KRAS_status",
    figsize=(15, 8),
    add_dots=True,
    dpi=300,
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=11)
plt.ylabel("Proportion of cells", fontsize=13, fontweight='bold')
plt.xlabel("Cell type", fontsize=13, fontweight='bold') 
plt.title("Cell type composition: mKRAS vs wtKRAS", fontsize=15, fontweight='bold', pad=20)
plt.savefig('./sccoda_boxplot_mKRAS_vs_wtKRAS.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
pt.pl.coda.stacked_barplot(
    sccoda_data, modality_key="coda", feature_name="KRAS_status", figsize=(6, 10)
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=11)
plt.ylabel("Proportion of cells", fontsize=13, fontweight='bold')
plt.xlabel("KRAS status", fontsize=13, fontweight='bold') 
plt.title("Cell type proportion: mKRAS vs wtKRAS", fontsize=15, fontweight='bold', pad=20)
plt.savefig('./Cell type proportion: mKRAS vs wtKRAS.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
sccoda_data = sccoda_model.prepare(
    sccoda_data,
    modality_key="coda",
    formula="KRAS_status",
    reference_cell_type="Smooth muscle cells", #в качестве референсного типа клеток стоит брать тот, что не изменяется
)
sccoda_model.run_nuts(sccoda_data, modality_key="coda", rng_key=1234)

In [ ]:
sccoda_model.set_fdr(sccoda_data, 0.3)

In [ ]:
sccoda_model.credible_effects(sccoda_data, modality_key="coda")

In [ ]:
pt.pl.coda.effects_barplot(sccoda_data, "coda", "KRAS_status", dpi = 300)
plt.title("scCODA Effects: mKRAS vs wtKRAS", fontsize=14, fontweight='bold')
plt.savefig('./sccoda_effects_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')

<H1>Дифференциальная экспрессия</H1>

In [ ]:
def DE_custom(group1, group2, cell_clu, text_fc, text_pval, legend):
    sadata = adata[((adata.obs.KRAS_status == group1) |(adata.obs.KRAS_status == group2))&
          (adata.obs['cell_type_refined'] == cell_clu)]
    #sadata = adata.raw.to_adata()

    def rank_genes_groups_df(adata, group, pval_cutoff : float =None, logfc_cutoff=None): 
        d = pd.DataFrame() 
        for k in ['scores', 'names', 'logfoldchanges', 'pvals', 'pvals_adj']: 
            d[k] = adata.uns["rank_genes_groups"][k][group] 
        if pval_cutoff is not None: 
            d = d[d["pvals_adj"] < pval_cutoff] 
        if logfc_cutoff is not None: 
            d = d[d["logfoldchanges"].abs() > logfc_cutoff] 
        return d

    sc.tl.rank_genes_groups(sadata, "KRAS_status", method = 'wilcoxon',corr_method="benjamini-hochberg")
    de_df = rank_genes_groups_df(sadata, group2)
    de_df = de_df.assign(nlog10 = (de_df['pvals']+1e-300).apply(lambda x:-np.log10(x)))

    def map_color(a):
        logfoldchanges, names, nlog10 = a

        if abs(logfoldchanges) < 2.5 or nlog10 < 2.5:
            return 'not_significant'
        return 'significant'

    de_df['color'] = de_df[['logfoldchanges', 'names', 'nlog10']].apply(map_color, axis = 1)
    
    de_df['namez'] = de_df['names']
    de_df = de_df.set_index('namez')
    #de_df = de_df.assign(average_expression = (sadata.to_df().sum() / 2000).sort_values() + 1)



    ax = sns.scatterplot(data = de_df, x = 'logfoldchanges', y = 'nlog10',
                        hue = 'color', hue_order = ['not_significant', 
                                                    'significant'],
                        palette = ['#E7E7E7', '#E87676'], legend = legend,
                         #size = 'average_expression', 
                         sizes = (40, 400))

    ax.axhline(2, zorder = 0, c = 'k', lw = 2, ls = '--')
    ax.axvline(1, zorder = 0, c = 'k', lw = 2, ls = '--')
    ax.axvline(-1, zorder = 0, c = 'k', lw = 2, ls = '--')



    texts = []
    for i in range(len(de_df)):
        if de_df.iloc[i].nlog10 > text_pval and abs(de_df.iloc[i].logfoldchanges) > text_fc and abs(de_df.iloc[i].logfoldchanges) < 10:
            texts.append(plt.text(x = de_df.iloc[i].logfoldchanges, y = de_df.iloc[i].nlog10, s = de_df.iloc[i].names,
                                 fontsize = 12, weight = 'bold'))
    adjust_text(texts)





    plt.legend(loc = 1, bbox_to_anchor = (1.4,1), frameon = False, prop = {'weight':'bold'})

    for axis in ['bottom', 'left']:
        ax.spines[axis].set_linewidth(2)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.tick_params(width = 2)

    plt.xticks(size = 12, weight = 'bold')
    plt.yticks(size = 12, weight = 'bold')

    plt.xlabel("$log_{2}$ fold change", size = 15)
    plt.ylabel("-$log_{10}$ FDR", size = 15)
    plt.title(group1 + ' vs ' + group2+' | cell type: '+cell_clu)

    plt.xlim(-12,12)

    plt.show()
    return(de_df)

In [ ]:
import adjustText
from adjustText import adjust_text

In [ ]:
plt.rcParams['figure.figsize'] = [8,12]
d = DE_custom('wtKRAS','mKRAS','Tumor cells', text_fc = 3, text_pval = 50, legend= True)

In [ ]:
d

In [ ]:
gene_list_KRAS = d.loc[(d.pvals < 0.01)&(d.logfoldchanges>4.0)].names.tolist()

In [ ]:
gene_list = ['IGKC','XIST','TFCP2','SH3BGRL3','SOX2','PCK1','PCK2','S100A4', 'ALDH4A1','NTSR1', 'CLCA1','SPINK1', 'CLCA1','MAP2K1', 'MAPK1', 'ETS1', 'FOS', 'DUSP1']

In [ ]:
sc.pl.heatmap(sadata, gene_list, standard_scale = 'var', 
              groupby='KRAS_status', show_gene_labels=True,  swap_axes=True, cmap = 'viridis', figsize = (6,6))

In [ ]:
gene_list_WT = d.loc[(d.pvals < 0.05)&(d.logfoldchanges<1.0)].names.tolist()

In [ ]:
sadata = adata[((adata.obs.KRAS_status == 'wtKRAS') |(adata.obs.KRAS_status == 'mKRAS'))&
      (adata.obs['cell_type_refined'] == 'Tumor cells')]

In [ ]:
sc.pl.heatmap(sadata, gene_list_KRAS, standard_scale = 'var', 
              groupby='KRAS_status', show_gene_labels=True,  swap_axes=True, cmap = 'viridis', figsize = (10,30))

## Over-representation analysis and Gen-Set Enrichment Analysis (поиск ассоциированных биологических процессов)

In [ ]:
import gseapy as gp

In [ ]:
enr_kras = gp.enrichr(gene_list=gene_list_KRAS,
                 gene_sets=['Reactome_2022', 'MSigDB_Hallmark_2020'],
                 organism='human',
                 outdir=None, # don't write to disk
                )

In [ ]:
enr_wt = gp.enrichr(gene_list=gene_list_WT,
                 gene_sets=['Reactome_2022', 'MSigDB_Hallmark_2020'],
                 organism='human',
                 outdir=None, # don't write to disk
                )

In [ ]:
enr_kras.results.head()

In [ ]:
enr_wt.results.head()

In [ ]:
from gseapy import barplot, dotplot

In [ ]:
ax = barplot(enr_kras.results,
              column="Adjusted P-value",
              group='Gene_set', # set group, so you could do a multi-sample/library comparsion
              size=10,
              top_term=7,
              figsize=(3,5),
              color=['darkred', 'darkblue'], # set colors for group
             )
plt.savefig('barplot_mutkras.png', dpi=300, bbox_inches='tight', facecolor='white')
ax = barplot(enr_wt.results,
              column="Adjusted P-value",
              group='Gene_set', # set group, so you could do a multi-sample/library comparsion
              size=10,
              top_term=7,
              figsize=(3,5),
              color=['darkred', 'darkblue'], # set colors for group
             )
plt.savefig('barplot_wtkras.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
from gseapy import enrichment_map
# return two dataframe
nodes, edges = enrichment_map(enr_kras.res2d)

In [ ]:
import networkx as nx

In [ ]:
edges.head(10)

In [ ]:
G = nx.from_pandas_edgelist(edges,
                            source='src_idx',
                            target='targ_idx',
                            edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

fig, ax = plt.subplots(figsize=(12, 12))

# init node cooridnates
pos=nx.layout.spiral_layout(G)
#node_size = nx.get_node_attributes()
# draw node
nx.draw_networkx_nodes(G,
                       pos=pos,
                       cmap=plt.cm.RdYlBu,
                       node_color= 'salmon',
                       node_size=list(nodes.Hits_ratio *1000))
# draw node label
nx.draw_networkx_labels(G,
                        pos=pos,
                        labels=nodes.Term.to_dict())
# draw edge
edge_weight = nx.get_edge_attributes(G, 'jaccard_coef').values()
nx.draw_networkx_edges(G,
                       pos=pos,
                       width=list(map(lambda x: x*10, edge_weight)),
                       edge_color='#2E86AB')
ax.axis('off')  # Убирает и рамку, и деления
ax.set_title('Network of Terms: KRASmut', fontsize=16) 
plt.savefig('network_graph.png', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
nodes, edges = enrichment_map(enr_wt.res2d)

In [ ]:
G = nx.from_pandas_edgelist(edges,
                            source='src_idx',
                            target='targ_idx',
                            edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])

fig, ax = plt.subplots(figsize=(12, 12))

nodes = nodes[nodes.index.isin(G.nodes())]
nodes = nodes.loc[list(G.nodes())]  # выравниваем порядок

# init node cooridnates
pos=nx.layout.spiral_layout(G)
#node_size = nx.get_node_attributes()
# draw node
nx.draw_networkx_nodes(G,
                       pos=pos,
                       cmap=plt.cm.RdYlBu,
                       node_color='salmon',
                       node_size=list(nodes.Hits_ratio *1000))
# draw node label
nx.draw_networkx_labels(G,
                        pos=pos,
                        labels=nodes.Term.to_dict())
# draw edge
edge_weight = nx.get_edge_attributes(G, 'jaccard_coef').values()
nx.draw_networkx_edges(G,
                       pos=pos,
                       width=list(map(lambda x: x*10, edge_weight)),
                       edge_color='#2E86AB')
ax.axis('off')  # Убирает и рамку, и деления
ax.set_title('Network of Terms:wtKRAS', fontsize=16) 
plt.savefig('network_graph_wtRKAS.png', dpi = 300, bbox_inches='tight')
plt.show()

## Межклеточная коммуникация

In [ ]:
import cell2cell

In [ ]:
import liana as li
from liana.mt import rank_aggregate

In [ ]:
sample_key = 'sample'
condition_key = 'KRAS_status'
groupby = 'cell_type_refined'

In [ ]:
li.mt.rank_aggregate.by_sample(
    adata,
    groupby=groupby,
    sample_key=sample_key, # sample key by which we which to loop
    use_raw=True, 
    verbose='full', # use 'full' to show all information
    n_perms=20, # reduce number of permutations for speed
    return_all_lrs=True, # return all LR values
    )

In [ ]:
import plotnine as p9
import cell2cell as c2c

In [ ]:
tensor = li.multi.to_tensor_c2c(adata,
                                sample_key=sample_key,
                                score_key='magnitude_rank', # can be any score from liana
                                how='outer_cells' # how to join the samples
                                )

In [ ]:
from collections import defaultdict

In [ ]:
disease_extent_key = 'disease_extent'
site_key = 'site'

In [ ]:
context_dict = adata.obs[[sample_key, condition_key]].drop_duplicates()
context_dict = dict(zip(context_dict[sample_key], context_dict[condition_key]))
context_dict = defaultdict(lambda: 'Unknown', context_dict)

tensor_meta = c2c.tensor.generate_tensor_metadata(interaction_tensor=tensor,
                                                  metadata_dicts=[context_dict, None, None, None],
                                                  fill_with_order_elements=True
                                                  )

In [ ]:
tensor = c2c.analysis.run_tensor_cell2cell_pipeline(tensor,
                                                    tensor_meta,
                                                    copy_tensor=True, # Whether to output a new tensor or modifying the original
                                                    rank=6, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis. Here, it was precomuputed.
                                                    tf_optimization='regular', # To define how robust we want the analysis to be. 
                                                    random_state=123, # Random seed for reproducibility
                                                    device='cuda',# Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                                    elbow_metric='error', # Metric to use in the elbow analysis.
                                                    smooth_elbow=False, # Whether smoothing the metric of the elbow analysis.
                                                    upper_rank=20, # Max number of factors to try in the elbow analysis
                                                    tf_init='random', # Initialization method of the tensor factorization
                                                    tf_svd='numpy_svd', # Type of SVD to use if the initialization is 'svd'
                                                    cmaps=None, # Color palettes to use in color each of the dimensions. Must be a list of palettes.
                                                    sample_col='Element', # Columns containing the elements in the tensor metadata
                                                    group_col='Category', # Columns containing the major groups in the tensor metadata
                                                    output_fig=True, # Whether to output the figures. If False, figures won't be saved a files if a folder was passed in output_folder.
                                                    output_folder='./figures/tensor_factors/'
                                                    )

In [ ]:
factors = tensor.factors

In [ ]:
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(factors, 
                                                                           sender_label='Sender Cells',
                                                                           receiver_label='Receiver Cells')

In [ ]:
network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby='receivers')

In [ ]:
import matplotlib.pyplot as plt


fig, (ax1,ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [4,4]}, figsize = (8,10))

sns.heatmap(factors['Sender Cells'], annot = True, fmt = ".1f", cmap = 'viridis', vmin = 0, vmax = 0.7,yticklabels = True, xticklabels = False, ax =ax1)
ax1.title.set_text('Sender Cells by factors')
ax1.set_ylabel('')
sns.heatmap(factors['Receiver Cells'], annot = True, fmt = ".1f",cmap = 'viridis', vmin = 0, vmax = 0.7,yticklabels = True, ax =ax2)
ax2.title.set_text('Receiver Cells by factors')
ax2.set_ylabel('')

plt.tight_layout()

plt.savefig('./figures/tensor_factors/sender_recivers.png', bbox_inches = 'tight')    
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [2,2]
_ = plt.hist(network_by_factors.values.flatten(), bins = 50)

In [ ]:
ccc_threshold = 0.01
c2c.plotting.ccc_networks_plot(factors,
                               included_factors=['Factor 1', 'Factor 2','Factor 3','Factor 4','Factor 5','Factor 6'],
                               ccc_threshold=ccc_threshold,
                               edge_color='blue',
                               node_color = '#F7C2C1',
                               network_layout = 'spring',
                               nrows=2,
                               edge_arrow_size = 50,
                               node_size=2000,
                               panel_size=(12,12), # This changes the size of each figure panel.
                              )
fig = plt.gcf()
fig.savefig('./figures/tensor_factors/ccc_networks_all_factors.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
lr_cm = c2c.plotting.loading_clustermap(factors['Ligand-Receptor Pairs'],
                                        loading_threshold=0.1, # To consider only top LRs
                                        use_zscore=True,
                                        method = 'ward',
                                        optimal_leaf=True,
                                        figsize=(14,7),
                                        cmap = 'bwr'
                                       )
fig = plt.gcf()
fig.savefig('./figures/tensor_factors/clustermap_Ligand-Receptor Pairs.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
meta.genetic_hormonal_features = meta.genetic_hormonal_features.map({'wtKRAS, mss':'wtKRAS','wtKRAS , mss':'wtKRAS',
                                                                              'mKRAS, mss':'mKRAS'})

In [ ]:
factors_table = factors['Contexts']
ann_table = meta.set_index('patient')
full_table = pd.concat([factors_table, ann_table], axis = 1)

In [ ]:
dict(zip(meta.patient, meta.disease_extent))

In [ ]:
boxplot = c2c.plotting.factor_plot.context_boxplot(factors['Contexts'],
                                                   metadict=context_dict,
                                                   nrows=2,
                                                   figsize=(12, 5),
                                                   statistical_test='Mann-Whitney',
                                                   pval_correction='benjamini-hochberg',
                                                   cmap='viridis',
                                                   verbose=False,
                                                  )
fig = plt.gcf()
fig.savefig('./figures/tensor_factors/factor_plot.context_boxplot.png', dpi=300, bbox_inches='tight', facecolor='white')

<H1>Hotspot</H1>

In [ ]:
import hotspot

In [ ]:
tumor = adata[adata.obs.cell_type_refined.isin(['Tumor cells'])]

In [ ]:
tumor = tumor.raw.to_adata()

In [ ]:
print(f"Тип данных в .X: {tumor.X.dtype}")

In [ ]:

sc.pp.highly_variable_genes(tumor, n_top_genes=2000,  flavor="seurat_v3",)

In [ ]:
tumor.raw = tumor

In [ ]:
tumor = tumor[:, tumor.var.highly_variable]

In [ ]:
sc.pp.pca(tumor)
sc.pp.neighbors(tumor)

In [ ]:
sc.external.pp.harmony_integrate(tumor, key = 'sample')

In [ ]:
tumor.obsm['X_pca'] = tumor.obsm['X_pca_harmony']
sc.pp.neighbors(tumor, n_pcs=30)
sc.tl.umap(tumor)

In [ ]:
sc.pl.umap(tumor, color = ['KRAS_status','cell_type_refined'])

In [ ]:
sc.tl.leiden(tumor, resolution= 0.4)

In [ ]:
plt.rcParams['figure.figsize'] = [5,5]
sc.pl.umap(tumor, color = 'leiden', legend_loc = 'on data', legend_fontsize = 'x-large') #color = 'label'

In [ ]:
hs = hotspot.Hotspot(
    tumor,
    model='danb',
    latent_obsm_key="X_pca",
)

In [ ]:
hs.create_knn_graph(
    weighted_graph=False, n_neighbors=30,
)

In [ ]:
hs_results = hs.compute_autocorrelations()

In [ ]:
hs_genes = hs_results.loc[hs_results.FDR < 0.05].sort_values('Z', ascending=False).head(200).index

In [ ]:
lcz = hs.compute_local_correlations(hs_genes)

In [ ]:
modules = hs.create_modules(min_gene_threshold=20, core_only=True, fdr_threshold=0.05)

In [ ]:
hs.plot_local_correlations()
plt.savefig('./hotspot_local_correlations.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
module_scores = hs.calculate_module_scores()

In [ ]:
tumor.obs = pd.concat([tumor.obs, module_scores.add_prefix('m_')], axis = 1)

In [ ]:
f_m = ['m_3','m_2','m_1','m_4']

In [ ]:
sc.pl.dotplot(tumor, f_m,
             standard_scale = 'var',
             groupby = 'leiden',
             swap_axes = True, figsize = (4,6))
plt.savefig('./hotspot_dotplot.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
plt.rcParams['figure.figsize'] = [6,6]
sc.pl.embedding(tumor, 
                basis = 'X_umap', 
                color = f_m,
                legend_loc = 'on data',
                ncols = 2,
               legend_fontsize = 10,
               legend_fontweight = 10,
               save='./hotspot_embedding.png')

In [ ]:
x = pd.DataFrame(modules)

for i in range(1,5):
    print('MODULE ', i)
    print(x.loc[x.Module == i].index.tolist())
    print('##################################')

In [ ]:
import gseapy as gp

In [ ]:
from gseapy import *

In [ ]:
import pandas as pd

def item_series(item, indexed=None):
    if indexed is not None:
        if hasattr(indexed, 'index'):
            return pd.Series([item]*len(indexed), index=indexed.index)
        elif type(indexed) is int and indexed > 0:
            return pd.Series([item]*indexed, index=np.arange(indexed))
    return pd.Series()

def pivot_vectors(vec1, vec2):
    name1 = str(vec1.name)
    name2 = str(vec2.name)
    sub_df = pd.DataFrame({name1: vec1,
                           name2: vec2})
    fill_dict = {}
    sub_df.fillna(value=fill_dict, inplace=True)
    sub_df = sub_df.assign(N=item_series(1, sub_df))

    return pd.pivot_table(data=sub_df, columns=name1,
                          index=name2, values='N', aggfunc=sum).fillna(0).astype(int)

def convert_to_shares(df):
    return df.div(df.sum())

In [ ]:
pivot_vectors(tumor.obs.KRAS_status, tumor.obs.leiden).plot.barh()

In [ ]:
sc.pl.heatmap(tumor, f_m,
             standard_scale = 'var',
             groupby = 'leiden',
             swap_axes = True, figsize = (16,3))

In [ ]:
module_genes = x.loc[x.Module.isin([4])].index.tolist() 

In [ ]:
import gseapy as gp

In [ ]:
enr_4 = gp.enrichr(gene_list=module_genes,
                 gene_sets=['Reactome_Pathways_2024','GO_Biological_Process_2023'],
                 # organism='human', # organism argment is ignored because user input a background
                )

In [ ]:
gp.plot.barplot(enr_4.results,
              top_term=10,
              figsize=(3,5),)
plt.savefig('./enrichment_module4_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
module_1_genes = x.loc[x.Module.isin([1])].index.tolist() 

In [ ]:
enr_1 = gp.enrichr(gene_list=module_1_genes,
                 gene_sets=['Reactome_Pathways_2024','GO_Biological_Process_2023'],
                 # organism='human', # organism argment is ignored because user input a background
                )

In [ ]:
gp.plot.barplot(enr_1.results,
              top_term=10,
              figsize=(3,5),)
plt.savefig('./enrichment_module1_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
module_2_genes = x.loc[x.Module.isin([2])].index.tolist() 

In [ ]:
enr_2 = gp.enrichr(gene_list=module_2_genes,
                 gene_sets=['Reactome_Pathways_2024','GO_Biological_Process_2023'],
                 # organism='human', # organism argment is ignored because user input a background
                )
gp.plot.barplot(enr_2.results,
              top_term=10,
              figsize=(3,5),)
plt.savefig('./enrichment_module2_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')

In [ ]:
module_3_genes = x.loc[x.Module.isin([3])].index.tolist() 

In [ ]:
enr_3 = gp.enrichr(gene_list=module_3_genes,
                 gene_sets=['Reactome_Pathways_2024','GO_Biological_Process_2023'],
                 # organism='human', # organism argment is ignored because user input a background
                )
gp.plot.barplot(enr_3.results,
              top_term=10,
              figsize=(3,5),)
plt.savefig('./enrichment_module3_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')

## Оценка обогащения для каждой клетки

In [ ]:
d

In [ ]:
import drug2cell as d2c
import blitzgsea as blitz

In [ ]:
targets = blitz.enrichr.get_library("Reactome_Pathways_2024")

In [ ]:
d2c.score(tumor, targets=targets, use_raw=True)

In [ ]:
procs = tumor.uns['drug2cell']

In [ ]:
sc.tl.rank_genes_groups(procs, groupby = 'KRAS_status')

In [ ]:
sc.pl.rank_genes_groups_dotplot(procs, values_to_plot='logfoldchanges',
                                cmap = 'bwr',
                                min_logfoldchange = 1,
                                swap_axes=True,           # Гены по Y, группы по X (вертикальный формат)
                                 var_group_rotation=1,
                                   figsize=(8, 5),
                                groupby = 'KRAS_status',
                                title="Pathway enrichment analysis in KRASmut and wt CRC cells",
                               save="_Pathway enrichment analysis in KRASmut and wt CRC cells.png")

In [ ]:
sc.pl.umap(procs, color = ['Diseases of the Neuronal System','Loss of Function of TGFBR1 in Cancer', 'KRAS_status'])

In [ ]:
d2c.score(adata, use_raw=True)

anti-EGFR

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'CETUX').columns.tolist()

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'PANIT').columns.tolist()

Иммунотерапия

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'PEMBRO').columns.tolist()

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'DOSTAR').columns.tolist()

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'NIVOLUMAB').columns.tolist()

KRAS ingibitors

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'SOTOR').columns.tolist()

anti-VEGF

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'REGO').columns.tolist()

In [ ]:
adata.uns['drug2cell'].to_df().filter(like = 'BEVAC').columns.tolist()

In [ ]:
sc.pl.umap(adata, color="cell_type_refined", color_map="OrRd", save="_cell_type_refined_OrRd.png")

In [ ]:
plt.rcParams['figure.figsize'] = [6,6]
sc.pl.umap(adata.uns['drug2cell'], color="CHEMBL1201577|CETUXIMAB", color_map="OrRd", save="_cetuximab_eff.png")

In [ ]:
sadata = adata[adata.obs.cell_type_refined == 'Tumor cells']

In [ ]:
sc.tl.rank_genes_groups(sadata.uns['drug2cell'], method="wilcoxon", groupby="KRAS_status")

In [ ]:
immunotherapy_names = ['CHEMBL2108738|NIVOLUMAB', 'CHEMBL3137343|PEMBROLIZUMAB','CHEMBL4298124|DOSTARLIMAB']

In [ ]:
anti_EGFR_names = ['CHEMBL1201577|CETUXIMAB', 'CHEMBL1201827|PANITUMUMAB']

In [ ]:
anti_VEGF = ['CHEMBL1946170|REGORAFENIB', 'CHEMBL1201583|BEVACIZUMAB']

In [ ]:
ingKRAS = ['CHEMBL4535757|SOTORASIB']

In [ ]:
sc.pl.rank_genes_groups_dotplot(sadata.uns['drug2cell'], var_names = anti_EGFR_names, swap_axes=True, groupby = 'KRAS_status',
                                values_to_plot="logfoldchanges", figsize = (4,2), title="anti-EGFR", save="_anti_EGFR_dotplot.png")

In [ ]:
sc.pl.rank_genes_groups_dotplot(sadata.uns['drug2cell'], var_names = immunotherapy_names, swap_axes=True, groupby = 'KRAS_status',
                                values_to_plot="logfoldchanges", figsize = (4,2),title="immunotherapy", save="_immunotherapy_dotplot.png")

In [ ]:
sc.pl.rank_genes_groups_dotplot(sadata.uns['drug2cell'], var_names = ingKRAS, swap_axes=True, groupby = 'KRAS_status',
                                values_to_plot="logfoldchanges", figsize = (4,2),title="KRAS(G12C)ing", save="_ingKRAS_dotplot.png")

In [ ]:
sc.pl.rank_genes_groups_dotplot(sadata.uns['drug2cell'], var_names = anti_VEGF, swap_axes=True, groupby = 'KRAS_status',
                                values_to_plot="logfoldchanges", figsize = (4,2),title="anti-VEGF", save="_anti_VEGF_dotplot.png")